# Baselines supervisées

Pour situer les approches à base d'auto-encodeur (notebooks 01 et 02), il me faut un point
de comparaison : que donnent des modèles supervisés classiques, entraînés directement sur
les données étiquetées ?

Je prends deux modèles standards sur données tabulaires — une régression logistique et une
random forest — et je les règle proprement par validation croisée (grid search). Ce sont
mes baselines : le but n'est pas de les battre à tout prix, mais d'avoir une référence
crédible pour interpréter les résultats des autres notebooks.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 123

## Données

Même dataset et même découpage que le notebook 02 (80 % train / 20 % test, stratifié) pour
que la comparaison soit juste. Le réglage des hyperparamètres se fait par validation croisée
à l'intérieur du train, donc le test reste totalement à part.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train :", X_train.shape, "| Test :", X_test.shape)

## Réglage par grid search

Chaque modèle est encapsulé dans un pipeline. La régression logistique a besoin d'un
`StandardScaler` (elle est sensible à l'échelle des variables) ; la random forest n'en a pas
besoin puisqu'elle travaille par seuils. Le scoring est le ROC-AUC, en validation croisée
stratifiée 5 folds.

In [ ]:
models = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(solver="liblinear", class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "RandomForest": Pipeline([
        ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
}

param_grids = {
    "LogisticRegression": {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__penalty": ["l1", "l2"],
    },
    "RandomForest": {
        "clf__n_estimators": [50, 100, 200],
        "clf__max_depth": [5, 10, None],
        "clf__min_samples_leaf": [1, 2, 4],
    },
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

best_models = {}
grid_results = []
for model_name, pipeline in models.items():
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1,
        refit=True,
    )
    grid_search.fit(X_train, y_train)
    best_models[model_name] = grid_search.best_estimator_
    grid_results.append({
        "Modele": model_name,
        "Best ROC-AUC (CV)": round(grid_search.best_score_, 4),
        "Meilleurs hyperparamètres": grid_search.best_params_,
    })

df_grid = pd.DataFrame(grid_results).set_index("Modele")
display(df_grid)

## Résultats sur le test

In [ ]:
rows = []
for model_name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    f1m = f1_score(y_test, y_pred, average="macro")

    print(f"{'='*20} {model_name} {'='*20}")
    print(f"ROC-AUC : {auc:.4f} | F1 (macro) : {f1m:.4f}\n")
    print(classification_report(y_test, y_pred, target_names=data.target_names))
    rows.append({"Modele": model_name, "ROC-AUC": round(auc, 4), "F1 (macro)": round(f1m, 4)})

pd.DataFrame(rows).set_index("Modele")

## Ce que je retiens

Sur ce dataset, les deux baselines sont très solides (ROC-AUC autour de 0.99). C'est ce à
quoi il faut confronter les approches auto-encodeur : la détection d'anomalie non supervisée
(notebook 01) reste logiquement en dessous, tandis que le SSL suivi de fine-tuning
(notebook 02) s'en approche nettement. Le tableau récapitulatif complet est dans le README.